# EDA — Atendimento e perfil do cliente
### Case NPS Preditivo | FIAP — Tech Challenge, Fase 1

**Responsável:** Felipe  
**Parte do grupo:** atendimento e perfil do cliente  
**Base utilizada:** `data/processed/desafio_nps_fase_1_clean.csv` (preparada em `01-cleansing.ipynb`)

---

### O que queremos entender?

Imagine que a empresa recebe uma nota baixa depois que o pedido foi concluído. O número, por si só, não explica o que deu errado. O objetivo desta análise é **investigar quais sinais do atendimento aparecem com mais frequência nas experiências negativas e verificar se existem diferenças importantes entre perfis de clientes**.

Vamos responder a quatro perguntas do desafio, dentro do nosso recorte:

1. **Quais fatores de atendimento estão mais associados à insatisfação?**
2. **O que caracteriza os pedidos avaliados por detratores?**
3. **Existe algum ponto em que a experiência parece piorar de forma acentuada?**
4. **Região, idade, tempo de relacionamento ou valor do pedido diferenciam a satisfação?**

Também vamos relacionar o NPS à recompra, porque isso ajuda a mostrar a importância do tema para o negócio.

> **Limite desta entrega:** este é um EDA de atendimento e perfil. A análise aprofundada de entrega e atrasos pertence aos colegas de logística. A integração acontecerá na versão final do grupo.

**Como ler o notebook:** cada seção parte de uma pergunta, apresenta os dados e termina com uma interpretação em linguagem de negócio. As relações encontradas são **associações**, não provas de causa e efeito.

## 1. Carregamento e conferência dos dados

O grupo já fez a limpeza no notebook `01-cleansing.ipynb`. **Não vamos repetir nem alterar essa limpeza aqui.** Antes de começar, vamos conferir a base que chegou até nós.

O caminho é buscado em duas posições comuns: rodando a análise na pasta `notebooks/` ou na raiz do repositório.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid', context='notebook', font_scale=1.0)
pd.set_option('display.max_columns', 35)
pd.set_option('display.float_format', lambda valor: f'{valor:,.2f}')

AZUL = '#326E96'
VERMELHO = '#B94B45'
VERDE = '#468771'
AMARELO = '#D2A34B'
CINZA = '#B8C0C6'
N_MIN = 30  # apenas referência prática para leitura; não é teste estatístico

caminhos = [
    Path('../data/processed/desafio_nps_fase_1_clean.csv'),
    Path('data/processed/desafio_nps_fase_1_clean.csv'),
]
caminho = next((p for p in caminhos if p.is_file()), None)
if caminho is None:
    raise FileNotFoundError(
        'Não encontrei a base tratada. Confira se o arquivo '
        'data/processed/desafio_nps_fase_1_clean.csv existe e, se necessário, '
        'execute primeiro o notebook 01-cleansing.ipynb.'
    )

df = pd.read_csv(caminho).copy()
necessarias = [
    'customer_id', 'order_id', 'nps_score', 'customer_region',
    'customer_age', 'customer_tenure_months', 'order_value',
    'complaints_count', 'customer_service_contacts', 'resolution_time_days',
    'repeat_purchase_30d',
]
ausentes = sorted(set(necessarias) - set(df.columns))
if ausentes:
    raise ValueError(f'A base não tem as colunas necessárias: {ausentes}')

print(f'Base utilizada: {caminho.as_posix()}')
print(f'Pedidos/registros: {len(df):,} | Colunas: {df.shape[1]}')
print(f'Clientes distintos: {df.customer_id.nunique():,}')
display(df[necessarias].head(5))

### 1.1. Qualidade dos campos utilizados

Uma análise bonita perde valor quando os dados são interpretados da maneira errada. Por isso, vamos verificar faltantes, duplicações e situações que merecem explicação do negócio.

**Unidade de análise:** trabalhamos por pedido/registro. Se o mesmo cliente fez mais de um pedido, os resultados não representam necessariamente o percentual de **pessoas únicas**.

In [ ]:
colunas_eda = [
    'nps_score', 'complaints_count', 'customer_service_contacts',
    'resolution_time_days', 'customer_region', 'customer_age',
    'customer_tenure_months', 'order_value', 'repeat_purchase_30d'
]
qualidade = pd.DataFrame({
    'valores_ausentes': df[colunas_eda].isna().sum(),
    '%_ausentes': (df[colunas_eda].isna().mean() * 100).round(2),
})
display(qualidade)

verificacoes = pd.Series({
    'order_id repetido': df.order_id.duplicated().sum(),
    'customer_id repetido': df.customer_id.duplicated().sum(),
    'Nota fora da escala de 0 a 10': (~df.nps_score.between(0, 10)).sum(),
    'Reclamações negativas': (df.complaints_count < 0).sum(),
    'Contatos negativos': (df.customer_service_contacts < 0).sum(),
    'Resolução negativa': (df.resolution_time_days < 0).sum(),
    'Recompra diferente de 0 ou 1': (~df.repeat_purchase_30d.isin([0, 1])).sum(),
    'Reclamação registrada sem contato no SAC': ((df.complaints_count > 0) & (df.customer_service_contacts == 0)).sum(),
    'Resolução > 0 sem reclamação': ((df.resolution_time_days > 0) & (df.complaints_count == 0)).sum(),
}, name='registros')
display(verificacoes.to_frame())

problemas_bloqueantes = (
    df[colunas_eda].isna().any().any()
    or (~df.nps_score.between(0, 10)).any()
    or (~df.repeat_purchase_30d.isin([0, 1])).any()
)
if problemas_bloqueantes:
    raise ValueError('Existem valores faltantes ou fora das regras centrais. Revisar o cleansing.')

print('Atenção: reclamação sem SAC não é necessariamente erro: pode existir outro canal.')
print('Também falta confirmar se 0 dia de resolução significa resolução imediata ou ausência de caso.')

## 2. Como vamos calcular o NPS?

A variável **`nps_score`** é a nota registrada depois da jornada de compra. Para responder às questões do case, vamos separar as avaliações em:

- **Detratores:** notas abaixo de 7;
- **Neutros:** notas de 7 até menos de 9;
- **Promotores:** notas iguais ou maiores que 9.

O **NPS agregado** é a porcentagem de promotores menos a porcentagem de detratores. Já a **nota média** é outro indicador: os dois não devem ser confundidos.

**Nota metodológica:** no NPS tradicional, a resposta é dada em valores inteiros de 0 a 10. Como esta base tem notas com casas decimais, mantivemos a nota original e adotamos os limites acima, coerentes com as faixas tradicionais. O grupo deve preservar a mesma convenção nos demais materiais.

In [ ]:
df['categoria_nps'] = np.select(
    [df.nps_score < 7, df.nps_score < 9],
    ['Detrator', 'Neutro'], default='Promotor'
)
ORDEM_NPS = ['Detrator', 'Neutro', 'Promotor']
contagens = df.categoria_nps.value_counts().reindex(ORDEM_NPS, fill_value=0)
pct_nps = 100 * contagens / len(df)
nps_geral = 100 * (contagens['Promotor'] - contagens['Detrator']) / len(df)

display(pd.DataFrame({'pedidos': contagens, 'participação_%': pct_nps.round(1)}))
fig, ax = plt.subplots(figsize=(9, 3.5))
cores = [VERMELHO, AMARELO, VERDE]
barras = ax.barh(ORDEM_NPS, pct_nps.to_numpy(), color=cores, height=.62)
ax.invert_yaxis()
ax.set_xlim(0, 112)
ax.set_xlabel('Participação dos pedidos (%)')
ax.set_title('Como os clientes avaliaram a experiência?', fontweight='bold')
for barra, n, pct in zip(barras, contagens, pct_nps):
    ax.text(pct + 1, barra.get_y() + barra.get_height()/2,
            f'{pct:.1f}%  (n={n})', va='center', fontsize=10)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

display(Markdown(
    f'**Em resumo:** a nota média é **{df.nps_score.mean():.2f}**, '
    f'a mediana é **{df.nps_score.median():.2f}** e o NPS agregado é '
    f'**{nps_geral:+.1f} pontos**. '
    f'Dos {len(df):,} registros, **{pct_nps["Detrator"]:.1f}%** estão '
    'na categoria de detratores. Esse é o cenário que vamos tentar entender.'
))

## 3. Como vamos comparar os grupos?

Para não enxergar uma tendência onde existem poucos pedidos, toda comparação terá **quantidade de registros (n)**, **nota média** e **percentual de detratores**. Também vamos mostrar o NPS por grupo, quando fizer sentido.

Grupos com **menos de 30 pedidos** aparecerão em cinza e precisam de leitura cautelosa. É uma referência prática de apresentação, **não uma garantia estatística de confiabilidade**. Evitamos gráficos com dois eixos diferentes porque eles podem exagerar ou esconder diferenças.

In [ ]:
def resumo_grupo(dados, coluna):
    """Resumo descritivo por grupo; não mede efeito causal."""
    r = dados.groupby(coluna, observed=True, dropna=False).agg(
        pedidos=('nps_score', 'size'),
        nota_media=('nps_score', 'mean'),
        detratores=('categoria_nps', lambda x: x.eq('Detrator').sum()),
        promotores=('categoria_nps', lambda x: x.eq('Promotor').sum()),
    )
    r['pct_detratores'] = 100 * r.detratores / r.pedidos
    r['nps_grupo'] = 100 * (r.promotores - r.detratores) / r.pedidos
    r['observacao'] = np.where(r.pedidos < N_MIN, 'amostra pequena', '—')
    return r


def apresentar_resumo(r):
    colunas = ['pedidos', 'nota_media', 'pct_detratores', 'nps_grupo', 'observacao']
    display(r[colunas].round(1).rename(columns={
        'nota_media': 'nota média', 'pct_detratores': 'detratores (%)',
        'nps_grupo': 'NPS do grupo', 'observacao': 'leitura'
    }))


def comparar_grupos(r, titulo):
    """Duas escalas separadas, com etiquetas da amostra."""
    if r.empty:
        print('Não há registros para esse agrupamento.'); return
    nomes = r.index.astype(str).tolist()
    y = np.arange(len(r))
    h = max(3.4, 2.0 + len(r) * 0.46)
    fig, axes = plt.subplots(1, 2, figsize=(13, h), sharey=True)
    c_nota = [AZUL if n >= N_MIN else CINZA for n in r.pedidos]
    c_det = [VERMELHO if n >= N_MIN else CINZA for n in r.pedidos]
    axes[0].barh(y, r.nota_media, color=c_nota, height=.67)
    axes[1].barh(y, r.pct_detratores, color=c_det, height=.67)
    for ax, lab, xmax in zip(axes, ['Nota média (0–10)', 'Detratores (%)'], [10, 115]):
        ax.set_yticks(y, nomes)
        ax.invert_yaxis()
        ax.set_xlim(0, xmax)
        ax.set_xlabel(lab)
        ax.grid(axis='y', visible=False)
        ax.grid(axis='x', alpha=.16)
        sns.despine(ax=ax, left=True)
    for idx, n in enumerate(r.pedidos):
        axes[1].text(min(float(r.pct_detratores.iloc[idx]) + 1.5, 110), idx,
                     f'n={int(n)}', va='center', fontsize=9)
    fig.suptitle(titulo, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
    if (r.pedidos < N_MIN).any():
        print('Barras cinza: menos de 30 pedidos. Interpretar com cautela.')

## 4. Reclamações: quando a insatisfação se intensifica?

**Pergunta:** clientes com mais reclamações também aparecem com notas mais baixas? E existe algum salto importante entre uma quantidade e outra?

Começamos por reclamações porque elas representam **problemas percebidos ou registrados**. Mas não sabemos apenas por essa variável qual foi o motivo, a gravidade ou o canal utilizado.

In [ ]:
# Mantemos 0 a 7 reclamações separadas; agregamos 8+ para evitar extremos muito raros.
df['grupo_reclamacoes'] = pd.Categorical(
    np.where(df.complaints_count >= 8, '8+', df.complaints_count.astype(int).astype(str)),
    categories=[str(i) for i in range(8)] + ['8+'], ordered=True
)
reclamacoes = resumo_grupo(df, 'grupo_reclamacoes')
apresentar_resumo(reclamacoes)
comparar_grupos(reclamacoes, 'Mais reclamações, pior avaliação?')

# A hipótese do segundo registro é avaliada em grupos separados, sem agregar demais.
por_quantidade = resumo_grupo(df, 'complaints_count')
if 1 in por_quantidade.index and 2 in por_quantidade.index:
    n1, n2 = int(por_quantidade.loc[1, 'pedidos']), int(por_quantidade.loc[2, 'pedidos'])
    t1, t2 = por_quantidade.loc[1, 'pct_detratores'], por_quantidade.loc[2, 'pct_detratores']
    diferenca_pp = t2 - t1
    display(Markdown(
        f'**Possível ponto de atenção — segunda reclamação:** a taxa de detratores '
        f'passou de **{t1:.1f}%** (1 reclamação, n={n1}) para '
        f'**{t2:.1f}%** (2 reclamações, n={n2}), '
        f'uma diferença de **{diferenca_pp:+.1f} pontos percentuais**. '
        'É um sinal para investigar, não uma prova de que a segunda reclamação causou a insatisfação.'
    ))

print(f'Correlação de Pearson entre número de reclamações e nota: {df.complaints_count.corr(df.nps_score):+.2f}')

**O que isso sugere para o negócio?**

Uma reclamação pode ser resolvida sem comprometer totalmente a experiência. Quando há reincidência, é importante descobrir **por que o problema continua acontecendo**. Um acompanhamento específico a partir da segunda reclamação é uma hipótese prática a testar.

**Cuidado:** pedidos sem reclamações são poucos na base original, e algumas quantidades altas também são raras. Além disso, um problema grave pode provocar simultaneamente várias reclamações e uma nota baixa — a EDA não consegue separar essas explicações.

## 5. Contatos com o atendimento: o cliente precisou insistir?

**Pergunta:** a satisfação muda quando o cliente entra em contato mais vezes?

Repetir contatos pode indicar que o problema não foi resolvido na primeira tentativa, **mas também** pode refletir casos mais complexos. Não devemos atribuir a culpa automaticamente ao SAC.

In [ ]:
df['grupo_contatos'] = pd.cut(
    df.customer_service_contacts, bins=[-0.1, .5, 1.5, 2.5, 3.5, np.inf],
    labels=['0', '1', '2', '3', '4+']
)
contatos = resumo_grupo(df, 'grupo_contatos')
apresentar_resumo(contatos)
comparar_grupos(contatos, 'Repetição de contatos e satisfação')

primeiro = contatos.loc['0'] if '0' in contatos.index else None
ultimo = contatos.loc['4+'] if '4+' in contatos.index else None
if primeiro is not None and ultimo is not None:
    display(Markdown(
        f'**Leitura:** com **nenhum contato**, a nota média foi '
        f'**{primeiro.nota_media:.2f}** (n={int(primeiro.pedidos)}). '
        f'Com **4 ou mais contatos**, a média foi **{ultimo.nota_media:.2f}** '
        f'(n={int(ultimo.pedidos)}). '
        'Isso sugere usar a repetição de contatos como sinal de atenção, '
        'sem concluir que o atendimento causou a nota menor.'
    ))
print(f'Correlação de Pearson entre contatos e nota: {df.customer_service_contacts.corr(df.nps_score):+.2f}')

**Como transformar em ação?**

Vale monitorar **contatos repetidos no mesmo pedido** e analisar os motivos de retorno. Para afirmar que houve falha na primeira resolução, precisaríamos de informações como *resolução no primeiro contato (FCR)* e motivo do chamado — essas variáveis não constam no dicionário fornecido.

Clientes sem contato também podem avaliar negativamente a compra. Por isso, **SAC silencioso não equivale a cliente satisfeito**.

## 6. Tempo de resolução: a demora faz diferença?

**Pergunta:** problemas que levam mais tempo para serem resolvidos aparecem associados a notas piores?

Vamos observar pedidos com **contato ou reclamação**. O significado do valor `resolution_time_days = 0` precisa ser confirmado: pode representar conclusão no mesmo dia ou um registro sem atendimento. Por isso, mostramos o grupo, mas evitamos tirar conclusões definitivas sobre ele.

In [ ]:
com_ocorrencia = df.loc[
    (df.customer_service_contacts > 0) | (df.complaints_count > 0)
].copy()
com_ocorrencia['grupo_resolucao'] = pd.cut(
    com_ocorrencia.resolution_time_days,
    bins=[-.1, .5, 1.5, 3.5, 7.5, np.inf],
    labels=['0 dia*', '1 dia', '2–3 dias', '4–7 dias', '8+ dias']
)
resolucao = resumo_grupo(com_ocorrencia, 'grupo_resolucao')
apresentar_resumo(resolucao)
comparar_grupos(resolucao, 'Tempo registrado para resolução')

corr_resolucao = com_ocorrencia.resolution_time_days.corr(com_ocorrencia.nps_score)
print(f'Correlação de Pearson entre tempo de resolução e nota nesse recorte: {corr_resolucao:+.2f}')
print('* O significado de 0 dia deve ser validado com o negócio.')

**Interpretação:** se os tempos maiores estiverem associados a notas menores, faz sentido acompanhar o prazo de resolução. Ainda assim, pode ser que **os casos mais graves sejam justamente os que demoram mais**. Não dá para afirmar que reduzir o prazo, sozinho, teria a mesma intensidade de efeito.

O próximo passo operacional seria cruzar a duração com **motivo e gravidade dos chamados**, quando esses dados estiverem disponíveis. Só então faria sentido propor um SLA específico por tipo de ocorrência.

## 7. Quando reclamações e contatos acontecem juntos

Os indicadores anteriores foram vistos separadamente. Agora vamos cruzar **reclamações × contatos** para enxergar onde se concentram os registros de maior insatisfação.

Esse cruzamento é todo de **atendimento**, respeitando a divisão de trabalho do grupo. As células com menos de 30 pedidos não entram na escala de cores: uma porcentagem calculada a partir de poucos casos pode ser enganosa.

In [ ]:
df['grupo_reclamacoes_cruzado'] = pd.cut(
    df.complaints_count, bins=[-.1, 1.5, 3.5, 5.5, np.inf],
    labels=['0–1', '2–3', '4–5', '6+']
)
cruzamento = df.groupby(
    ['grupo_contatos', 'grupo_reclamacoes_cruzado'], observed=True
).agg(
    pedidos=('nps_score', 'size'),
    detratores=('categoria_nps', lambda x: x.eq('Detrator').sum())
).reset_index()
cruzamento['taxa_detratores'] = 100 * cruzamento.detratores / cruzamento.pedidos
matriz_n = cruzamento.pivot(index='grupo_contatos', columns='grupo_reclamacoes_cruzado', values='pedidos')
matriz_taxa = cruzamento.pivot(index='grupo_contatos', columns='grupo_reclamacoes_cruzado', values='taxa_detratores')
matriz_n = matriz_n.reindex(index=df.grupo_contatos.cat.categories,
                            columns=df.grupo_reclamacoes_cruzado.cat.categories).fillna(0).astype(int)
matriz_taxa = matriz_taxa.reindex(index=matriz_n.index, columns=matriz_n.columns)
matriz_visivel = matriz_taxa.mask(matriz_n < N_MIN)

print('Quantidade de pedidos em cada combinação:')
display(matriz_n)
fig, ax = plt.subplots(figsize=(8.8, 4.7))
sns.heatmap(matriz_visivel, annot=True, fmt='.0f', cmap='Reds', vmin=0, vmax=100,
            linewidths=.6, cbar_kws={'label': 'Detratores (%)'}, ax=ax)
ax.set_title('Percentual de detratores por reclamações e contatos', fontweight='bold')
ax.set_xlabel('Reclamações no pedido')
ax.set_ylabel('Contatos com SAC')
plt.tight_layout()
plt.show()
print('Casas vazias: ausência de registros ou grupo com menos de 30 pedidos.')

**O que procurar no mapa:** grupos em que o cliente precisou reclamar várias vezes **e** entrar em contato repetidamente. Esses casos merecem investigação prioritária — com a ressalva de que reclamação e contato não são medidas independentes.

A análise não mostra o que veio primeiro. Sem datas de cada evento, não podemos afirmar uma sequência como “reclamou → contatou → avaliou mal”.

## 8. Perfil do cliente: quem avalia melhor ou pior?

Até aqui, olhamos para sinais da experiência. Agora a pergunta muda: **quem é o cliente faz muita diferença?**

Vamos comparar região, idade, valor do pedido e tempo de relacionamento. O cuidado principal é não exagerar diferenças pequenas ou transformar médias de grupos em regras sobre indivíduos.

In [ ]:
df['faixa_idade'] = pd.cut(
    df.customer_age, bins=[-np.inf, 29, 39, 49, 59, np.inf],
    labels=['Até 29', '30–39', '40–49', '50–59', '60+']
)
df['faixa_valor'] = pd.cut(
    df.order_value, bins=[-np.inf, 200, 400, 600, 1000, np.inf],
    labels=['Até R$ 200', 'R$ 200–400', 'R$ 400–600', 'R$ 600–1.000', 'Acima de R$ 1.000']
)
df['faixa_tempo'] = pd.cut(
    df.customer_tenure_months, bins=[-np.inf, 12, 36, 60, np.inf],
    labels=['Até 1 ano', '1–3 anos', '3–5 anos', '5+ anos']
)

perfis = {
    'Região': 'customer_region',
    'Idade': 'faixa_idade',
    'Valor do pedido': 'faixa_valor',
    'Relacionamento': 'faixa_tempo'
}
resumos_perfil = {}
for titulo, coluna in perfis.items():
    resumos_perfil[titulo] = resumo_grupo(df, coluna)
    print(f'--- {titulo} ---')
    apresentar_resumo(resumos_perfil[titulo])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (titulo, r) in zip(axes.flat, resumos_perfil.items()):
    labels = r.index.astype(str)
    cor_barras = [VERMELHO if n >= N_MIN else CINZA for n in r.pedidos]
    bars = ax.barh(labels, r.pct_detratores, color=cor_barras, height=.67)
    ax.axvline(pct_nps['Detrator'], ls='--', lw=1.2, color='#3D4A57', alpha=.7)
    ax.set_xlim(0, 113)
    ax.invert_yaxis()
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Detratores (%)')
    for b, n in zip(bars, r.pedidos):
        ax.text(min(b.get_width()+1, 109), b.get_y()+b.get_height()/2,
                f'n={int(n)}', va='center', fontsize=8)
    sns.despine(ax=ax, left=True)
fig.suptitle('Taxa de detratores por perfil | linha tracejada = média geral',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Amplitude da nota média entre os grupos com n >= 30.
# É uma comparação descritiva, não um ranking de causalidade.
linhas = []
for titulo, r in resumos_perfil.items():
    aptos = r.loc[r.pedidos >= N_MIN, 'nota_media']
    if len(aptos) > 1:
        linhas.append({
            'indicador': titulo,
            'menor média': aptos.min(),
            'maior média': aptos.max(),
            'diferença em pontos': aptos.max() - aptos.min(),
        })
variacoes_perfil = pd.DataFrame(linhas).sort_values('diferença em pontos', ascending=False)
display(variacoes_perfil.round(2))
if not variacoes_perfil.empty:
    maior = variacoes_perfil.iloc[0]
    display(Markdown(
        f'**Leitura:** entre os recortes de perfil, a maior diferença observada '
        f'foi em **{maior["indicador"]}** (**{maior["diferença em pontos"]:.2f} ponto(s)** '
        'entre as médias dos grupos analisados). Compare esse valor com as diferenças '
        'encontradas nos indicadores de atendimento, sem interpretar isso como efeito causal.'
    ))

**O que esse bloco ajuda a responder?**

Se as regiões e faixas de perfil apresentarem valores muito próximos, **não há motivo para afirmar que existe um “tipo de cliente problemático”** só por idade, região ou faixa de gasto. Pode ser mais útil observar **o que aconteceu com o pedido** do que quem fez a compra.

Também não devemos concluir que perfil “não importa”: as médias podem esconder diferenças dentro de cada grupo. A leitura correta é **“nesta amostra, nesses recortes, a diferença observada é pequena ou grande”**, conforme a tabela calculada.

## 9. Satisfação e recompra: por que isso importa para a empresa?

O desafio também pede uma reflexão sobre a importância do NPS para a **recompra**. Por isso, vamos olhar para a variável `repeat_purchase_30d`, que indica se houve nova compra em até 30 dias.

**Atenção:** a recompra é registrada **depois do pedido**. Ela serve aqui para descrever associação entre experiência e comportamento posterior. **Não pode ser usada como entrada de um modelo que tenta prever o NPS antes da pesquisa.**

In [ ]:
recompra = df.groupby('categoria_nps').agg(
    pedidos=('repeat_purchase_30d', 'size'),
    recompras=('repeat_purchase_30d', 'sum')
).reindex(ORDEM_NPS, fill_value=0)
recompra['taxa_recompra'] = 100 * recompra.recompras / recompra.pedidos.replace(0, np.nan)
display(recompra.round(1).rename(columns={'taxa_recompra': 'recompra (%)'}))

fig, ax = plt.subplots(figsize=(8, 4))
barras = ax.bar(recompra.index, recompra.taxa_recompra, color=[VERMELHO, AMARELO, VERDE])
ax.set_ylim(0, 113)
ax.set_ylabel('Recompra em 30 dias (%)')
ax.set_title('Recompra observada por categoria de NPS', fontweight='bold')
for barra, n, taxa in zip(barras, recompra.pedidos, recompra.taxa_recompra):
    if pd.notna(taxa):
        ax.text(barra.get_x()+barra.get_width()/2, taxa+2, f'{taxa:.1f}%  (n={int(n)})',
                ha='center', fontsize=9)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

min_nota_recomprador = df.loc[df.repeat_purchase_30d.eq(1), 'nps_score'].min()
print(f'Recompras registradas: {int(df.repeat_purchase_30d.sum())} de {len(df)} pedidos')
print(f'Menor nota entre os pedidos associados a recompra: {min_nota_recomprador}')

if (recompra.loc['Detrator', 'taxa_recompra'] == 0 and
        recompra.loc['Promotor', 'taxa_recompra'] == 100):
    display(Markdown(
        '**Ponto de atenção:** encontramos **0% de recompra entre detratores** '
        'e **100% entre promotores**. É um padrão muito extremo para generalizar. '
        'Vale perguntar como a base foi produzida e se existe alguma regra de geração '
        'ligando `repeat_purchase_30d` ao `nps_score`.'
    ))

**Como explicar para uma liderança?**

A recompra pode ajudar a mostrar **o custo de uma experiência ruim**, mas este conjunto de dados sozinho não prova que melhorar a nota fará um cliente voltar a comprar. Não temos um experimento controlado nem a linha do tempo detalhada.

Como as taxas nos extremos são muito fortes na execução original, é prudente tratar esse resultado como **uma característica da base utilizada**, não como estimativa de comportamento do mercado inteiro.

## 10. Quais situações deveriam chamar a atenção primeiro?

Uma taxa de detratores de 100% em dois pedidos é menos útil para priorização do que uma taxa um pouco menor em centenas de pedidos. Aqui vamos cruzar:

- **volume de pedidos** no grupo;
- **taxa de detratores** dentro dele;
- **participação no total de detratores**.

Os limites abaixo são exemplos de triagem *exploratória*, não uma regra operacional validada nem um modelo preditivo.

In [ ]:
# Três faixas mutuamente exclusivas, criadas apenas para facilitar a priorização.
severo = (df.complaints_count >= 4) & (df.customer_service_contacts >= 3)
recorrente = df.complaints_count >= 2

df['situacao_atendimento'] = np.select(
    [severo, recorrente],
    ['4+ reclamações e 3+ contatos', '2+ reclamações (demais pedidos)'],
    default='Até 1 reclamação'
)
prioridade = resumo_grupo(df, 'situacao_atendimento')
prioridade['parte_dos_pedidos_%'] = 100 * prioridade.pedidos / len(df)
prioridade['parte_dos_detratores_%'] = (
    100 * prioridade.detratores / df.categoria_nps.eq('Detrator').sum()
)
prioridade = prioridade.sort_values('detratores', ascending=False)
display(prioridade[[
    'pedidos', 'parte_dos_pedidos_%', 'detratores', 'pct_detratores',
    'parte_dos_detratores_%', 'nota_media'
]].round(1).rename(columns={
    'pct_detratores': 'taxa detratores (%)',
    'parte_dos_detratores_%': '% de todos os detratores',
    'parte_dos_pedidos_%': '% de pedidos'
}))

fig, ax = plt.subplots(figsize=(10, 4))
mostra = prioridade.sort_values('parte_dos_detratores_%')
b = ax.barh(mostra.index, mostra['parte_dos_detratores_%'], color=AZUL)
ax.set_xlim(0, max(105, mostra['parte_dos_detratores_%'].max()+10))
ax.set_xlabel('Participação entre todos os detratores (%)')
ax.set_title('Onde está concentrado o volume de experiências negativas?', fontweight='bold')
for barra, valor in zip(b, mostra['parte_dos_detratores_%']):
    ax.text(valor+1, barra.get_y()+barra.get_height()/2, f'{valor:.1f}%', va='center')
sns.despine(ax=ax, left=True)
plt.tight_layout()
plt.show()

maior_grupo = prioridade['detratores'].idxmax()
display(Markdown(
    f'**O maior volume de detratores está em:** {maior_grupo}. '
    'Isso ajuda a orientar uma investigação de causa-raiz. '
    'O resultado descreve registros históricos, não uma previsão individual.'
))

## 11. O que pode ser feito com esses resultados?

A EDA não resolve os problemas operacionais por si só. Ela ajuda a indicar **onde investigar primeiro** e **como medir se a ação está funcionando**.

| Sinal observado | Ação para testar | Indicador para acompanhar |
|---|---|---|
| Reclamações recorrentes | Investigar motivo e desfecho da segunda reclamação; priorizar retorno ao cliente | Pedidos com 2+ reclamações, taxa de detratores |
| Vários contatos para o mesmo pedido | Revisar histórico antes de responder; testar acompanhamento até a conclusão | Contatos por pedido, retornos, possível FCR quando disponível |
| Prazo longo para solucionar problemas | Separar chamados por tipo/gravidade e revisar fluxos demorados | Tempo por tipo de problema, SLA, taxa de detratores |
| Diferenças pequenas entre perfis amplos | Evitar ações baseadas só em idade ou região; focar no histórico de problemas | NPS e detratores por perfil e por ocorrência |
| Registros inconsistentes ou ambíguos | Validar definição de contato, reclamação e tempo zero | Completude e consistência dos dados |

**Exemplo de hipótese testável:** oferecer acompanhamento prioritário a pedidos que chegam à segunda reclamação e comparar, ao longo do tempo, os resultados dessa ação com um grupo comparável. **Só assim seria possível começar a avaliar impacto**, e não apenas associação.

## 12. Limitações e cuidados para as próximas fases

Esta seção é tão importante quanto os gráficos: mostra o que podemos e o que **não podemos** concluir.

### O que exige cuidado

**1. Associação não é causalidade.** Reclamações, contatos e notas baixas podem ser consequências de um mesmo problema. Não sabemos a ordem exata dos eventos.

**2. Data leakage (vazamento de informação).** O desafio fala em prever o NPS **antes da pesquisa**. Uma variável só poderá entrar em um futuro modelo se seu valor estiver disponível **no instante da previsão** — não basta existir no CSV histórico.

| Campo | Nesta análise exploratória | Em uma previsão antes da pesquisa |
|---|---|---|
| `nps_score` | É a variável que queremos explicar | **Alvo**, nunca entrada |
| `repeat_purchase_30d` | Serve para estudar associação com recompra | **Excluir**: resultado posterior |
| `csat_internal_score` | Não utilizamos neste EDA | **Não incluir sem auditoria**: origem e momento de coleta desconhecidos |
| `complaints_count` | Permitido para descrever histórico | Só reclamações registradas **até** a previsão |
| `customer_service_contacts` | Permitido para investigar atendimento | Só contatos registrados **até** a previsão |
| `resolution_time_days` | Permitido para análise retrospectiva | Só se a resolução já tiver ocorrido **antes** da previsão; caso contrário, pode vazar informação |
| Idade, região, tempo de cliente e valor do pedido | Comparações descritivas | Verificar se já eram conhecidos no instante de previsão |

**3. Resolução com zero dia e canais de reclamação.** Os valores não têm uma definição operacional completa. Uma reclamação sem contato no SAC pode ter sido feita por outro canal; não apagamos esses casos arbitrariamente.

**4. Recompra com padrão extremo.** Taxas muito próximas de 0% e 100% podem refletir regras de construção da base; não devem ser extrapoladas para um e-commerce real sem validação.

**5. Unidade de análise e pesquisa.** As taxas são por registro. Além disso, não sabemos se a pesquisa foi enviada e respondida por todos os compradores; isso pode produzir viés de seleção.

**6. Amostras pequenas.** Diferenças nas pontas das distribuições podem variar muito. O limite de 30 apenas ajuda na leitura e não substitui intervalos de confiança.

**7. Dados que faltam.** Motivo, canal e gravidade das reclamações, datas dos contatos, histórico de resolução e primeiro contato resolvido seriam fundamentais para uma análise mais forte.

> **Decisão para esta entrega:** não treinamos modelo, não fazemos separação treino/teste e não usamos a recompra como preditora. O objetivo desta parte é investigar os dados e entregar hipóteses gerenciais para a consolidação do grupo.

## 13. Resumo final — contribuição do Felipe

A FIAP pede respostas compreensíveis para alguém que gerencia a operação e não precisa dominar estatística. Por isso, a conclusão abaixo **é montada a partir dos dados executados neste notebook**. Ela não depende de valores escritos manualmente que podem envelhecer.

Na consolidação, vale levar **os gráficos mais claros e dois ou três achados principais**, sem tentar apresentar todas as tabelas.

In [ ]:
# Conclusão calculada com os dados efetivamente carregados.
linha_sinais = (
    f'**Reclamações:** a correlação com a nota foi '
    f'**{df.complaints_count.corr(df.nps_score):+.2f}**; '
    'a tabela da seção 4 permite observar como o risco varia por quantidade.'
)
linha_contatos = (
    f'**Contatos com o atendimento:** a correlação com a nota foi '
    f'**{df.customer_service_contacts.corr(df.nps_score):+.2f}**. '
    'A repetição pode funcionar como alerta de casos que merecem acompanhamento.'
)
linha_tempo = (
    f'**Prazo de resolução:** no recorte com contato ou reclamação, '
    f'a correlação foi **{corr_resolucao:+.2f}**. '
    'O significado do zero e a gravidade dos casos precisam ser validados.'
)

if not variacoes_perfil.empty:
    amp_min, amp_max = variacoes_perfil['diferença em pontos'].min(), variacoes_perfil['diferença em pontos'].max()
    linha_perfil = (
        f'**Perfil:** as diferenças entre médias dos grupos de perfil ficaram '
        f'entre **{amp_min:.2f} e {amp_max:.2f} pontos**, considerando os grupos '
        'com pelo menos 30 registros. Isso é uma comparação descritiva.'
    )
else:
    linha_perfil = '**Perfil:** amostras insuficientes para comparar os grupos propostos.'

texto_final = (
    '### O que aprendemos?\n\n'
    + linha_sinais + '\n\n'
    + linha_contatos + '\n\n'
    + linha_tempo + '\n\n'
    + linha_perfil + '\n\n'
    + '**Ponto que vale investigar:** a segunda reclamação e os pedidos com '
      'múltiplos contatos são boas hipóteses para monitoramento, sempre '
      'respeitando o momento em que cada dado foi registrado.\n\n'
    + '**Recomendação de negócio:** começar pela causa das reclamações recorrentes, '
      'avaliar o processo de atendimento e medir o resultado de qualquer mudança '
      'antes de afirmar que a ação melhorou a satisfação.\n\n'
    + '**Limitação central:** o EDA encontra associações, não comprova causas '
      'nem garante que as mesmas taxas apareçam fora desta base.'
)
display(Markdown(texto_final))